In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from tqdm.notebook import tqdm

from xcube.core.store import new_data_store
from xcube.core.chunk import chunk_dataset
from xcube.core.gridmapping import GridMapping
from xcube.core.geom import mask_dataset_by_geometry
from xcube_resampling.spatial import resample_in_space
from xcube_resampling.gridmapping import GridMapping

In [2]:
INPUT_DIR = "input_irrigation"

In [3]:
irr_store = new_data_store("file", root=INPUT_DIR)

In [4]:
bbox = [-5, 40, 3, 44] # Ebro Basin
time_range = ("2020-01-01", "2021-12-31")
# time_range = ("2020-01-01", "2020-01-31")

In [5]:
def split_date_range(start_date, end_date, num_days=30):
    if isinstance(start_date, str):
        start_date = datetime.strptime(start_date, "%Y-%m-%d")
    if isinstance(end_date, str):
        end_date = datetime.strptime(end_date, "%Y-%m-%d")
    
    result = []
    current = start_date
    
    while current <= end_date:
        chunk_end = min(current + timedelta(days=num_days - 1), end_date)
        result.append((current.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))
        current = chunk_end + timedelta(days=1)
    
    return result

In [6]:
time_ranges = split_date_range(time_range[0], time_range[1], 5)
time_ranges, len(time_ranges)

([('2020-01-01', '2020-01-05'),
  ('2020-01-06', '2020-01-10'),
  ('2020-01-11', '2020-01-15'),
  ('2020-01-16', '2020-01-20'),
  ('2020-01-21', '2020-01-25'),
  ('2020-01-26', '2020-01-30'),
  ('2020-01-31', '2020-01-31')],
 7)

In [7]:
era_store = new_data_store("file", root="era5")

In [8]:
%%time
# ERA5

cds_store = new_data_store("cds", normalize_names=True)

data_id = 'reanalysis-era5-land'
variables = ("soil_temperature_level_1", "potential_evaporation", "total_precipitation", "snowfall")
spatial_res = 0.1

for _time_range in tqdm(time_ranges):
    cds_cube = cds_store.open_data(
        data_id,
        cds_store.get_data_opener_ids()[0],
        variable_names=variables,
        bbox=bbox,
        spatial_res=spatial_res,
        time_range=_time_range
    )
    era_store.write_data(cds_cube, f"era5-{_time_range[0].replace("-", "_")}-{_time_range[1].replace("-", "_")}.zarr", replace=True)

  0%|          | 0/7 [00:00<?, ?it/s]

xcube-cds version 1.1.0.dev0
2025-09-02 14:34:16,132 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-09-02 14:34:17,541 INFO Request ID is be3d1469-0ce0-494d-b90c-7be4cd0f5fd2
2025-09-02 14:34:17,945 INFO status has been updated to accepted
2025-09-02 14:34:33,513 INFO status has been updated to running
2025-09-02 14:34:41,222 INFO status has been updated to accepted
2025-09-02 14:34:52,692 INFO status has been updated to running
2025-09-02 14:36:14,047 INFO status has been updated to successful


7bb00d04a7ba3c2d3154772c88e0a3a5.zip:   0%|          | 0.00/2.01M [00:00<?, ?B/s]

xcube-cds version 1.1.0.dev0
2025-09-02 14:38:04,796 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-09-02 14:38:05,055 INFO Request ID is 1f008b62-c966-4cc5-a84c-46d039c72b4a
2025-09-02 14:38:05,443 INFO status has been updated to accepted
2025-09-02 14:38:15,426 INFO status has been updated to running
2025-09-02 14:39:23,978 INFO status has been updated to successful


5954da27e2141f0a74b0a29717fdc296.zip:   0%|          | 0.00/2.00M [00:00<?, ?B/s]

...

xcube-cds version 1.1.0.dev0
2025-09-02 14:53:29,949 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-09-02 14:53:30,623 INFO Request ID is 1b2daa07-e00a-4459-916d-139dba93e99a
2025-09-02 14:53:31,031 INFO status has been updated to accepted
2025-09-02 14:53:46,023 INFO status has been updated to running
2025-09-02 14:54:49,880 INFO status has been updated to successful


776bcf2e5b731f4c5d752ca2a0b7dfec.zip:   0%|          | 0.00/2.37M [00:00<?, ?B/s]

xcube-cds version 1.1.0.dev0
2025-09-02 14:56:53,608 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-09-02 14:56:54,726 INFO Request ID is 3273a8af-c8ea-4fb8-bb6f-d8e92a27c67a
2025-09-02 14:56:54,808 INFO status has been updated to accepted
2025-09-02 14:57:03,558 INFO status has been updated to running
2025-09-02 14:57:25,289 INFO status has been updated to successful


f9208353dc2d10ba6b1f3bcf986075c5.zip:   0%|          | 0.00/76.3k [00:00<?, ?B/s]

CPU times: user 1.67 s, sys: 453 ms, total: 2.12 s
Wall time: 23min 12s


In [9]:
data_ids = era_store.list_data_ids()

In [10]:
%%time
from zappend.api import zappend

final_data_id = "era5.zarr"
config = {
    "target_dir": f"{INPUT_DIR}/era5.zarr",
    "force_new": True,
    "logging": True,
    "excluded_variables": ["expver", "number"]
}
zappend((f'era5/{data_id}' for data_id in sorted(data_ids)), config=config)

2025-09-02 14:57:28,648 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2025-09-02 14:57:28,654 INFO Opening slice dataset from era5/era5-2020_01_01-2020_01_05.zarr
2025-09-02 14:57:28,661 INFO Creating target dataset input_irrigation/era5.zarr
/home/yogesh/Projects/BC/irrigation-processor/.pixi/envs/default/lib/python3.12/site-packages/zappend/processor.py:151: FutureWarning: zarr_version is deprecated, use zarr_format
  target_ds.to_zarr(
2025-09-02 14:57:28,689 INFO Transaction completed.
2025-09-02 14:57:28,689 INFO Slice dataset era5/era5-2020_01_01-2020_01_05.zarr closed
2025-09-02 14:57:28,697 INFO Opening slice dataset from era5/era5-2020_01_06-2020_01_10.zarr
2025-09-02 14:57:28,707 INFO Updating target dataset input_irrigation/era5.zarr
2025-09-02 14:57:28,743 INFO Transaction completed.
2025-09-02 14:57:28,744 INFO Slice dataset era5/era5-2020_01_06-2020_01_10.zarr closed
2025-09-02 14:57:28,755 INFO Opening slice dataset 

CPU times: user 419 ms, sys: 65.7 ms, total: 484 ms
Wall time: 409 ms


7

In [11]:
irr_store.list_data_ids()

['era5.zarr', 'landcover2020global.zarr']

In [12]:
irr_store.open_data("era5.zarr")

<xarray.Dataset> Size: 37MB
Dimensions:  (time: 721, lat: 40, lon: 80)
Coordinates:
  * lat      (lat) float64 320B 43.95 43.85 43.75 43.65 ... 40.25 40.15 40.05
  * lon      (lon) float64 640B -4.95 -4.85 -4.75 -4.65 ... 2.65 2.75 2.85 2.95
  * time     (time) datetime64[ns] 6kB 2020-01-01 ... 2020-01-31
Data variables:
    pev      (time, lat, lon) float32 9MB dask.array<chunksize=(120, 40, 80), meta=np.ndarray>
    sf       (time, lat, lon) float32 9MB dask.array<chunksize=(120, 40, 80), meta=np.ndarray>
    stl1     (time, lat, lon) float32 9MB dask.array<chunksize=(120, 40, 80), meta=np.ndarray>
    tp       (time, lat, lon) float32 9MB dask.array<chunksize=(120, 40, 80), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    history:                 2025-09-02T12:35 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts